# Twelve70 styling prediction model error analysis

This notebook provides confusion matrix heatmaps, confidence vs. empirical accuracy plots, and detailed error mode distributions.
All numbers are computed from real model predictions on the held-out test split.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Validation Report Metrics

In [ ]:
with open('error_analysis_report.json') as f:
    report = json.load(f)

print("=== Error Analysis Summary ===")
print(f"Total Test Predictions : {report['total_predictions']}")
print(f"Correct Predictions    : {report['correct']}")
print(f"Incorrect Predictions  : {report['incorrect']}")
print(f"Empirical Top-1 Accuracy: {report['accuracy']:.2%}")
print(f"Confidence Calibration : {report['confidence_analysis']['confidence_calibration']}")

## 2. Top Confusion Pairs

In [ ]:
conf_pairs_df = pd.DataFrame(report['top_confusion_pairs'])
print(conf_pairs_df)

## 3. Confusion Matrix Heatmap (Top 15 Most Confused Colors)

In [ ]:
cm = pd.read_csv('confusion_matrix.csv', index_col=0)
# Exclude diagonal to focus strictly on mistakes
cm_errs = cm.copy()
for col in cm_errs.columns:
    if col in cm_errs.index:
        cm_errs.loc[col, col] = 0

top_err_colors = cm_errs.sum(axis=1).sort_values(ascending=False).head(15).index
filtered_cm = cm.loc[top_err_colors, top_err_colors]

plt.figure(figsize=(12, 10))
sns.heatmap(filtered_cm, annot=True, fmt='d', cmap='Oranges', cbar=True)
plt.title('Confusion Matrix Heatmap (Top 15 Colors by Misclassifications)')
plt.xlabel('Predicted Color')
plt.ylabel('Actual Color')
plt.tight_layout()
plt.savefig('confusion_matrix_heatmap.png', dpi=300)
plt.show()

## 4. Confidence vs. Empirical Accuracy Calibration

In [ ]:
cal_bins = report['confidence_analysis']['calibration_bins']
bin_labels = [b['bin'] for b in cal_bins]
cal_accuracies = [b['empirical_accuracy'] if b['empirical_accuracy'] is not None else 0.0 for b in cal_bins]
counts = [b['count'] for b in cal_bins]

plt.figure(figsize=(8, 6))
bars = plt.bar(bin_labels, cal_accuracies, color='lightcoral', edgecolor='black', alpha=0.8)
for bar, n in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'n={n}', ha='center', fontsize=9)
plt.plot([-0.5, 4.5], [0.1, 0.9], color='darkred', linestyle='--', label='Theoretical Ideal Calibration')
plt.title('Confidence Calibration Curve (Real Test Predictions)')
plt.xlabel('Prediction Confidence Range')
plt.ylabel('Empirical Accuracy')
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig('confidence_vs_accuracy.png', dpi=300)
plt.show()

## 5. Error Distribution Across Outfit Categories

In [ ]:
categories = ['Monochromatic Outfits', 'Contrast Outfits', 'Rare Color Combos']
accuracies = [
    report['error_patterns']['monochromatic_outfits']['accuracy'] or 0.0,
    report['error_patterns']['contrast_outfits']['accuracy'] or 0.0,
    report['error_patterns']['rare_color_combinations']['accuracy'] or 0.0
]

plt.figure(figsize=(8, 5))
plt.barh(categories, accuracies, color=['seagreen', 'steelblue', 'indianred'], edgecolor='black', height=0.6)
plt.axvline(x=report['accuracy'], color='black', linestyle=':', label=f"Global Mean Accuracy ({report['accuracy']:.1%})")
plt.title('Model Accuracy across Outfit Categories')
plt.xlabel('Accuracy')
plt.xlim(0, 1.0)
plt.legend()
plt.tight_layout()
plt.savefig('error_by_outfit_category.png', dpi=300)
plt.show()